## <center><span style="color:green">Flask</span> UI for <span style="color:blue">Gemini</span> Resume Optimizer</span></center>
***
#### Where the <span style="color:gold">Gradio</span> UI was used for the <span style="color:orange">OpenAI</span> / <span style="color:orange">ChatGPT</span> optimizer, the following notebook does the same for our <span style="color:blue">Gemini</span> optimizer.
***
### Step 1: Imports
> #### <span style="color:red">Standard imports as other notebooks, only this time we're importing the functions from the Python file ('<span style="color:firebrick">gemini_flash_functions</span>'), as well as <span style="color:green">Flask</span> and its following libraries</span>:
> ##### 1.) "<u><span style="color:green">Flask</u></span>" - to create the web app object;
> ##### 2.) "<u><span style="color:green">render_template</u></span>" - renders HTML templates (this is a webapp, after all);
> ##### 3.) "<u><span style="color:green">request</u></span>" - deals with the job description inputs that the users copy and paste into the program;
> ##### 4.) "<u><span style="color:green">flash</u></span>" - creates temporary, one-time messages for user interaction;
> ##### 5.) "<u><span style="color:green">redirect</u></span>" - redirects user to an actual URL so they don't have to reload the webapp after every interaction;
> ##### 6.) "<u><span style="color:green">url_for</u></span>" - converts functions to actual URL paths inside the <span style="color:green">Flask</span> application; and 
> ##### 7.) "<u><span style="color:green">session</u></span>" - lets you input multiple job descriptions w/o needing to reload the page.

#### One other added import: '<span style="color:green">tempfile</span>'. '<span style="color:green">Template</span>' is a <span style="color:blue">Python</span> module that creates temporary files and directories. It's used to:
> ##### a.) create an actual temporary Markdown file - the downloaded resume - to the browser; and
> ##### b.) it sets up a temporary path to that file and writes the optimized resume to that file.

In [1]:
# from flask import Flask, render_template, request, flash, redirect, url_for, send_file # jsonify(?)
# import os
# from dotenv import load_dotenv
# import google.generativeai as genai
# from markdown import markdown
# import tempfile

# # import the functions from 'gemini_flash_functions' python file:
# from gemini_flask_functions import (
#     configure_gemini_api,
#     tailored_resume_function_schema,
#     gemini_initialization_with_function_calling,
#     generate_tailoring_prompt,
#     get_gemini_response_with_function_calling
# )

#### Didn't like the look of the the code above (<span style="color:red"><b>^^^</b></span>). That's a pretty loaded import schedule for '<span style="color:blue">gemini_flask_functions</span>,' and we can import the entire thing with the simple import statement: '<span style="color:green ; font-family:menlo"><b>from gemini_flask_functions import *</b></span>.'

In [2]:
# Cleaner Imports
from flask import Flask, render_template, request, flash, redirect, url_for, session
import os
from dotenv import load_dotenv
import google.generativeai as genai
from markdown import markdown
import tempfile
import logging  # Import the logging module
from gemini_flask_functions import *


In [3]:
# Load environment variables from our .env file
load_dotenv()

True

In [4]:
# start Flask
app = Flask(__name__)

#### When you use an app in <span style="color:green">Flask</span>, it stores your information in cookies for the length of time you're using that app. Because we're dealing with your information and data here, it's best to secure things with a "<span style="color:red">SECRET_KEY</span>" set up. That way, when you launch the app, you have a "secret key" to lock in and protect the data you input into the app.
>##### <b>That key is stored in our environment variables for security's sake. It's being assigned the variable value "<span style="color:green ; font-family:menlo">app.secret_key</span> for later use. With the app in production, it will draw upon the "<span style="color:firebrick ; font-family:menlo">FLASK_SECRET_KEY</span>," but for development purposes, we're defaluting that value to "<span style="color:green ; font-family:menlo">dev</span>."</b>

In [5]:
# Bonus: this also avoids the secret key error at launch!
app.secret_key = os.environ.get("FLASK_SECRET_KEY", "dev")

#### If you've ever pointed your garage door opener at your garage, clicked the button, and watched in amazement... <i>as nothing happened</i>, you'll appreciate the fact that Python has a built-in logging function we can call on to give us some help on figuring out why that garage door didn't open.

##### <b><span style="color:firebrick ; font-family:menlo">"%(asctime)s - %(levelname)s - %(message)s"</span></b> breaks down like this:
>##### <b><span style="color:firebrick ; font-family:menlo">%(asctiem)s</span></b> == the time the issue occured, <b><span style="color:firebrick ; font-family:menlo">%(levelname)s</b></span> == how big a deal it is, and <b><span style="color:firebrick ; font-family:menlo">%(message)s</b></span> == what Python has to say about it.

In [6]:
# Configure logging
logging.basicConfig(level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

#### Next step is to set up a directory where we can send the uploaded <span style="color:green">Flask</span> files, ie: your resume.

In [7]:
app.config["UPLOAD_FOLDER"] = "uploads"
os.makedirs(app.config["UPLOAD_FOLDER"], exist_ok=True)

#### Time to set up the function schema. See the '<span style="color:coral">res_optimizer_new_and_improved_gemini</span>' notebook. For now, it's basically you designing the response you want back from the '<span style="color:blue">gemini-2.0-flash</span>' engine.

In [8]:
# Set up the function schema:
function_schema = {
    "name": "tailor_resume",
    "description": "Make my resume fit the job description so I can get a damn recruiter on the phone.",
    "parameters": {
        "type": "object",
        "properties": {
            "tailored_resume": {
                "type": "string",
                "description": "Your Markdown-formatted resume - for easy editing.",
            },
            "additional_suggestions": {
                "type": "string",
                "description": "Not bad! But here are some ideas to make your resume stronger for this position.",
            },
        },
        "required": ["tailored_resume"]
    }
}


#### Now to give <span style="color:blue">gemini-2.0-flash</span> an object to play with (what we're calling here, '<b><span style="color:firebrick ; font-family:menlo">gemini_model</span></b>').

In [9]:
# Inialize the Gemini 2.0 Flash Engine
gemini_model = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    tools=[{"function_declarations": [function_schema]}]
)

#### Add the prompt we engineered earlier, through the trial and error outlined in the '<span style="color:coral">res_optimizer_new_and_improved_gemini</span>' notebook.

In [10]:
# Throw in the Prompt aimed at the resume and job description
# added directive to run the tailor_resume function on 5/28 - kept getting inconsistent responses
def generate_tailoring_prompt(resume_text, jd_text):
    return f"""
You are a professional resume optimization expert.
Optimize the resume I have provided you to align with the given job description following the guidelines below:

1. Make the resume one page, relevant, keyword optimized, action-driven, and in reverse chronological order.
2. Format it cleanly in **Markdown**.
3. Optimize for Applicant Tracking Systems (ATS).
4. End with an "**Some Suggestions To Make Your Resume Stronger**" section if relevant.
5. You MUST call the tailor_resume function below with your final answer. Do NOT return plain text. Use the schema provided.


Resume:
{resume_text}

Job Description:
{jd_text}
"""

#### Before kicking off the <span style="color:green">Flask</span> app, you need to set up and initialize - "<i>give value to</i>" - the <span style="color:blue">Gemini API</span>.

#### The '<b><span style="color:green ; font-family:menlo">try</span></b>' block interacts with the <span style="color:blue">gemini-2.0-flash</span> engine and prepares it for our use. 
#### The '<b><span style="color:green ; font-family:menlo">except</span></b>' block tells us what to do if something goes wrong during the '<b><span style="color:green ; font-family:menlo">try</span></b>' block's execution, like an issue with the API key or model name, for instance. This block allows the app to keep running, just without the use of <span style="color:blue">gemini-2.0-flash's</span> capabilities.

In [12]:
# Configure Gemini API on app startup
try:
    configure_gemini_api()
    function_schema = tailored_resume_function_schema()
    gemini_model = gemini_initialization_with_function_calling(function_schema)
    logger.info("Gemini model initialized.")
except ValueError as e:
    logger.error(f"Error during initialization: {e}")
    #  Set gemini_model to None so the app can still run (with reduced functionality) - so the app doesn't crash.
    gemini_model = None

2025-05-18 15:58:26,274 - INFO - Gemini model initialized.


#### Now we need a function that rifles through the response we get from <span style="color:blue">Gemini</span> and looks for ("<b><span style="color:green">returns<span></b>") the '<b><span style="color:black ; font-family:menlo">tailored resume</span></b>' it was working on, as well as any of its '<b><span style="color:black ; font-family:menlo">additional_suggestions</span></b>.'

In [13]:
def parse_gemini_response(response):
    """
    This function goes through Gemini's answer (the 'response') and looks for the tailored resume and any additional suggestions.
    We just want to get values back - we don't want to write it to a file just yet - changes still may be made.
    """
    tailored_resume = None
    additional_suggestions = None

    if response.candidates:
        first_candidate = response.candidates[0]

        if first_candidate.content and first_candidate.content.parts:
            first_part = first_candidate.content.parts[0]

            if first_part.function_call:
                function_call = first_part.function_call

                if function_call.name == "tailor_resume":
                    arguments = function_call.args
                    tailored_resume = arguments.get("tailored_resume")
                    additional_suggestions = arguments.get("additional_suggestions")
            elif first_part.text:
                # Fallback if function calling didn't work
                tailored_resume = first_part.text
    
    # output had markdown fences (e.g., "```markdown")
    # if block below cleans up the output

    if tailored_resume and tailored_resume.strip().startswith("```markdown") and tailored_resume.strip().endswith("```"):
        # remove the "```markdown" at the beginning
        tailored_resume = tailored_resume.strip()[len("```markdown"):].strip()
        # remove ```fence from the end
        tailored_resume = tailored_resume.rsplit("```", 1)[0].strip()

    return tailored_resume, additional_suggestions

#### When <span style="color:green">Flask</span> launches, it will take the user to a website - a URL path. 
#### We want things to happen once the app opens on that path (indicated by the <b><span style="color:firebrick ; font-family:menlo">"/"</span></b>), so we use a decorator (the '<span style="color:darkorchid">@</span>' symbol) to tell <span style="color:green">Flask</span> that URL opens, it needs to run the <i>very next function</i> (here, the '<b><span style="color:blue ; font-family:menlo">index</span></b>' function) that follows. 

In [14]:
@app.route("/")
def index():
    """
    Render a home page with form for the user to upload their resume and a job description.
    """
    return render_template("index.html")

#### Now, for the meat and potatoes of everything. 

#### Using another decorator, we throw in some methods: '<b><span style="color:firebrick ; font-family:menlo">GET</span></b>' (you telling <span style="color:blue">Gemini</span> you want something), and '<b><span style="color:firebrick ; font-family:menlo">POST</span></b>' (<span style="color:blue">Gemini</span> going "Here you go!")

#### Just like before, the decorator tells <span style="color:green">Flask</span> to run the '<b><span style="color:blue ; font-family:menlo">tailor-resume</span></b>' function once the user is taken to the URL path, this one ending with <b><span style="color:firebrick ; font-family:menlo">"tailor_resume</span></b>.<b><span style="color:firebrick ; font-family:menlo">"</span></b>

#### '<span style="color:blue">tailor_resume</span>()' is a pretty loaded function telling Flask to do a few things:
> ##### 1.) The block beginning with '<b><span style="color:green ; font-family:menlo">if</span></b> <b><span style="font-family:menlo">request.</span></b><b><span style="color:blue ; font-family:menlo">method</span></b> <b><span style="color:darkmagenta ; font-family:menlo">==</span></b> <b><span style="color:firebrick ; font-family:menlo">"POST"</span>:</b>' takes in the job description text, checks for and tries to read any uploaded resume files, saves it for the session, and shows an error message if no resume is present;
> ##### 2.) '<b><span style="font-family:menlo">prompt</span></b><b><span style="color:darkmagenta ; font-family:menlo">=</span></b> <b><span style="font-family:menlo">generate_tailoring_prompt(resume_text, jd_text)</span></b>' utilizes the <b><span style="color:blue ; font-family:menlo">generate_tailoring_prompt</span></b> function from earlier, putting our engineered prompt up against the job description and our resume;
> ##### 3.) the '<b><span style="color:green ; font-family:menlo">try</span></b> / <b><span style="color:green ; font-family:menlo">except</span></b> block following sends the request to the <span style="color:blue">Gemini</span> API, initializes the model, and tells the model to generate content based on the instructions found inside the '<b><span style="font-family:menlo">contents</span></b>' and '<b><span style="font-family:menlo">tools</span></b>' brackets. It also tells <span style="color:green">Flask</span> what to do if there are any problems communicating with the API;
> ##### 4.) the last part (from '<b><span style="font-family:menlo">tailored_resume</span></b>' and '<b><span style="font-family:menlo">additional_suggestions</span></b>' through to the end '<b><span style="color:green ; font-family:menlo">return</span></b>' statement) deals with taking the <span style="color:blue">Gemini</span> response (if there is one!), checks if the <span style="color:blue">tailor_resume</span>' function was used, and extracts the arguments 'tailored_resume' and 'additional_suggestions.' The '<b><span style="color:green ; font-family:menlo">elif</span></b>' is used if the returned <b><span style="font-family:menlo">tailored_resume</span></b>' is just plain text - no function was used; and
> ##### 5.) Finally, it <b><span style="color:green ; font-family:menlo">returns</span></b> (displays) the rendered HTML template <b><span style="color:firebrick ; font-family:menlo">"result.html"</span></b> with the '<b><span style="font-family:menlo">tailored_resume</span></b>' and any '<b><span style="font-family:menlo">additional_suggestions</span></b>.'

In [15]:
@app.route("/tailor-resume", methods=["GET", "POST"])
def tailor_resume():
    """Process the resume and job description to generate a tailored resume"""
    # handle the resume upload:
    if request.method == "POST":
        jd_text = request.form.get("job_description")
        resume_text = ""

        # keep the uploaded resume going
        if "resume_on_file" not in session:
            file = request.files.get("resume_file")
            if file and file.filename:
                resume_text = file.read().decode("utf-8")
                session["resume_text"] = resume_text
                session["resume_on_file"] = True
            else:
                flash("Hey, dipshit: we still need your resume.")
                return redirect(url_for("index"))
        else:
            resume_text = session.get("resume_text")

# Build out the prompts and make the Gemini call
    prompt = generate_tailoring_prompt(resume_text, jd_text)

    try:
        response = gemini_model.generate_content(
            contents = [{"role" : "user", "parts" : [{"text" : prompt}]}], 
            tools = [{"function_declarations" : [function_schema]}],
         generation_config = genai.types.GenerationConfig(temperature = 0.7)
    )

    except Exception as exception:
        flash(f"We found a Gemini API error: {exception}")
        return redirect(url_for("index"))

    # Get the response from the function call
    tailored_resume = ""
    additional_suggestions = ""

    if response.candidates:
        first_part = response.candidates[0].content.parts[0]
        if first_part.function_call and first_part.function_call.name == "tailor_resume":
            args = first_part.function_call.args
            tailored_resume = args.get("tailored_resume", "")
            additional_suggestion = args.get("additional_suggestions", "")
        elif first_part.text:
            tailored_resume = first_part.text
    
    return render_template("result.html", tailored_resume_html = tailored_resume,
                           tailored_resume_md = tailored_resume, additional_suggetions = additional_suggestions
                           )

#### Next, it's important to clear the resume so that the user has control over which resume they want to use for that particular job descripton. This is also helpful in the app's scaling (as more users use it - tell your friends!), and in dumping your data from the app itself.

In [16]:
@app.route("/clear-resume")
def clear_resume():
    session.pop("resume_text", None)
    session.pop("resume_on_file", None)
    flash("We got rid of your resume to keep your data safe. Please upload a new one.")
    # redirect the browser to the home page 
    return redirect(url_for("index"))

#### This is all for naught if the user can't <b><span style="color:firebrick ; font-family:menlo">"/download"</span></b> their resume.

#### The <b><span style="color:blue ; font-family:menlo">download_resume</span></b> function that follows does a couple of things:
> ##### 1.) It takes the data - again, if there is any - from the submitted form ('<b><span style="font-family:menlo">request.</span><span style="color:blue ; font-family:menlo">form</span><span style="font-family:menlo">.</span><span style="color:blue ; font-family:menlo">get</span></b>') and stores it in a variable called '<b><span style="font-family:menlo">resume_md</span></b>;
> ##### 2.) It creates a temporary file on the <span style="color:green">Flask</span> server, then writes the tailored resume text into that temporary file using <b><span style="color:firebrick ; font-family:menlo">"utf-8"</span></b>, a universal encoding <i>language</i>(?) format computers can understand;
> ##### 3.) The file is then sent to the user's browser for download as a Markdown file, along with a suggested filename (here, '<b><span style="color:firebrick ; font-family:menlo">"your_editable_markdown_resume.md"</span></b>); and
> ##### 4.) <b><span style="color:green ; font-family:menlo">finally</span></b>, the temporary file is deleted from the server for security's sake and to keep things clean.

In [17]:
@app.route("/download-resume", methods = ["POST"])
def download_resume():
    resume_md = request.form.get("tailored_resume_md")
    if not resume_md:
        flash("There's no resume data to download.")
        # back to the homepage with you!
        return redirect(url_for("index"))
    # set 'delete' to 'False' so the temp file is not deleted and give it a '.md' extension for Markdown
    with tempfile.NamedTemporaryFile(delete = False, suffix = ".md") as temp:
        temp.write(resume_md.encode("utf-8"))
        temp_path = temp.name
    
    try:
        return send_file(temp_path, as_attachment = True,
                         download_name = "your_editable_markdown_resume.md", mimetype = "text/markdown"
                         )
    
    finally:
        os.unlink(temp_path)

#### At last, we can to launch the <span style="color:green">Flask</span> app.

#### '<b><span style="color:green ; font-family:menlo">if </span><span style="font-family:menlo">__name__</span> <span style="color:blue ; font-family:menlo">== </span><span style="color:firebrick ; font-family:menlo">"__main__"</span>:</b>' checks to make sure they <span style="color:blue">Python</span> script is being executed directly. If that's not the case, the app won't run. 
#### '<b><span style="font-family:menlo">port</span></b>' tries to get a port number from anywhere on your computer (the <b><span style="color:firebrick ; font-family:menlo">"0.0.0.0"</span></b>) - a doorway on your computer that lets your apps communicate over the internet. If no port is found, this app defaults to <span style="color:green">5000</span>, an access point that doesn't require admin privileges (you can use it without having to enter a Username and Password on your own computer).

In [18]:
if __name__ == "__main__":
    port = int(os.environ.get("PORT", 5000))
    app.run(host = "0.0.0.0", port = port, debug = True)

 * Serving Flask app '__main__'
 * Debug mode: on


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
On macOS, try disabling the 'AirPlay Receiver' service from System Preferences -> General -> AirDrop & Handoff.


SystemExit: 1

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### <span style="color:red">vvvvv</span> Failed / Unused <span style="color:green">Flask</span> code and Markdown below. <span style="color:red">vvvvv</span>

### <span style="color:red">Ruh-roh.</span>
#### <span style="color:green">Flask</span> starts up fine as the webapp, but keeps erroring out on a Markdown filter not being found whenever we try to actually use it.
#### After some homework, it looks like that's because <span style="color:green">Flask</span> uses <span style="color:firebrick">Jinja2</span> for its HTML templating, and <span style="color:firebrick">Jinja2</span> doesn't have any sort of Markdown filter. So, we have to make the conversion in the <span style="color:green">Flask</span> route itself.
> ##### Reviewing the code below, the Markdown <i>was</i> converted to HTML (the '<span style="color:hotpink">tailored_resume_html</span> = <span style="color:blue">markdown</span>(<span style="color:forestgreen">tailored_resume</span>)' line), so the problem may rest in the '<b>templates</b>' directory (in <span style="color:blue">VS Code</span>).
> ##### After running the app in the app in its virtual environment, <span style="color:steelblue">Mac Terminal</span> hit me with some more details, signalling an error in the 'Additional Suggestions' result.

In [19]:
# # call the api
# GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
# if not GEMINI_API_KEY:
#     raise ValueError("GEMINI_API_KEY was not loaded from your environment.")
# genai.configure(api_key=GEMINI_API_KEY)

In [20]:
# # app = Flask(__name__)
# # app.secret_key = os.environ.get("SECRET_KEY", "dev-secret-key")  # Change this in production

# from flask import Flask, render_template, request, flash, redirect, url_for, send_file
# import os
# import tempfile
# from dotenv import load_dotenv
# from markdown import markdown

# # Grab the functions_for_gemini_flask module
# from functions_for_gemini_flask import (
#     configure_gemini_api,
#     tailored_resume_function_schema,
#     gemini_initialization_with_function_calling,
#     generate_tailoring_prompt,
#     get_gemini_response_with_function_calling
# )

# # Load environment variables
# load_dotenv()

# app = Flask(__name__)
# app.secret_key = os.environ.get("SECRET_KEY", "dev-secret-key")  # Change this in production



In [22]:
# #  Declare that you're using the global variable

#     if not gemini_model:
#         flash("API configuration mistake. Please check your environment variables and restart the application.", "error")
#         logger.error("Gemini model is not initialized.")
#         return redirect(url_for("index"))

#     # Get resume and job description from form
#     resume_text = request.form.get("resume_text")
#     jd_text = request.form.get("job_description")

In [23]:
#     if not resume_text or not jd_text:
#         flash("Please provide both resume and job description.", "error")
#         return redirect(url_for("index"))


In [ ]:
 try:
#         # Generate tailored resume
#         prompt = generate_tailoring_prompt(resume_text, jd_text)
#         logger.debug(f"Generated prompt: {prompt}")  # Log the prompt

#         response = get_gemini_response_with_function_calling(
#             gemini_model,
#             prompt,
#             tailored_resume_function_schema() # Pass the schema here
#         )
#         logger.debug(f"Gemini API response: {response}")  # Log the full response

#         tailored_resume, additional_suggestions = parse_gemini_response(response)

#         if not tailored_resume:
#             flash("Failed to generate tailored resume. Please try again.", "error")
#             logger.warning("Gemini failed to generate resume.")
#             return redirect(url_for("index"))

#  Need to find out how to convert Markdown to HTML for display

# @app.route("/download-resume", methods=["POST"])
# def download_resume():
#     """Download the tailored resume as a markdown file"""
#     tailored_resume = request.form.get("tailored_resume_md")

#     if not tailored_resume:
#         flash("No resume data to download.", "warning")
#         return redirect(url_for("index"))

#     # Create a temporary file
#     try:
#         with tempfile.NamedTemporaryFile(delete=False, suffix=".md") as temp:
#             temp_path = temp.name
#             temp.write(tailored_resume.encode("utf-8"))
#         logger.info(f"Created temporary file: {temp_path}")

#         # Send the file to the user
#         return send_file(

# #     try:
# #         get_ipython
# #         # If we're in IPython/Jupyter, don't use the auto-reloader
# #         print("Flask app is ready. Access it at http://127.0.0.1:5000/")
# #         #  No app.run() here
# #     except NameError:
# #         # Normal Python environment
# #         app.run(debug=True)  #  <----  ONLY ONE app.run()
# #     #  No app.run() here
